# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/hands_on/DAY_1_HANDS_ON_SESSION_1_langgraph_web_search.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





<font color="red" size="10">
<b>HANDS-ON TIME: 15 mins</b>
</font>

# Hands-on: Stateful LangGraph with one tool

This notebook shows the minimal pattern:

1. Keep conversation state across turns.
2. Let an LLM call exactly one tool when needed.
3. Execute the tool and return to the LLM for final answer.

It uses:
- **LangGraph** for state + routing
- **One tool only**: `oai_web_search` (OpenAI Responses API `web_search`)

In [1]:
!pip install -q langgraph langchain-openai openai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 9.6 MB/s eta 0:00:00


In [2]:
import os
import json
from typing import TypedDict, Annotated, List

from dotenv import load_dotenv
from openai import OpenAI

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

In [3]:
# Load API key from .env or environment
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not found.")
    key = input("Enter OPENAI_API_KEY: ").strip()
    os.environ["OPENAI_API_KEY"] = key

print("OPENAI_API_KEY is set.")

OPENAI_API_KEY is set.


In [4]:
def _obj_get(x, key, default=None):
    if isinstance(x, dict):
        return x.get(key, default)
    return getattr(x, key, default)


@tool("oai_web_search")
def oai_web_search(query: str) -> dict:
    """Run OpenAI Responses API web_search and return text + citations + sources."""
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    response = client.responses.create(
        model="gpt-5",
        tools=[{"type": "web_search"}],
        include=["web_search_call.action.sources"],
        input=query,
    )

    output_text = _obj_get(response, "output_text", "") or ""
    citations = []
    sources = []

    for item in _obj_get(response, "output", []) or []:
        item_type = _obj_get(item, "type")

        if item_type == "message":
            for part in _obj_get(item, "content", []) or []:
                for ann in _obj_get(part, "annotations", []) or []:
                    if _obj_get(ann, "type") == "url_citation":
                        citations.append(
                            {
                                "title": _obj_get(ann, "title"),
                                "url": _obj_get(ann, "url"),
                            }
                        )

        if item_type == "web_search_call":
            action = _obj_get(item, "action")
            for src in _obj_get(action, "sources", []) or []:
                sources.append(
                    {
                        "title": _obj_get(src, "title"),
                        "url": _obj_get(src, "url"),
                    }
                )

    return {
        "query": query,
        "answer": output_text,
        "citations": citations,
        "sources": sources,
    }

In [5]:
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

In [6]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools([oai_web_search], tool_choice="auto")

SYSTEM_RULES = (
    "You can use only one tool: oai_web_search. "
    "For each user turn, call the tool at most once if needed, then provide a final concise answer. "
    "Do not perform repeated searches unless the user explicitly asks."
)


def llm_node(state: AgentState) -> AgentState:
    msgs = [SystemMessage(content=SYSTEM_RULES), *state["messages"]]
    ai = llm.invoke(msgs)
    return {"messages": [ai]}


def route_after_llm(state: AgentState):
    last = state["messages"][-1]
    calls = getattr(last, "tool_calls", None) or []
    return "tools" if calls else END


tool_node = ToolNode(tools=[oai_web_search])

workflow = StateGraph(AgentState)
workflow.add_node("llm", llm_node)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "llm")
workflow.add_conditional_edges("llm", route_after_llm, {"tools": "tools", END: END})
workflow.add_edge("tools", "llm")

app = workflow.compile(checkpointer=MemorySaver())

In [7]:
config = {"configurable": {"thread_id": "hands-on-web-search"}, "recursion_limit": 6}

# Turn 1: latest AI-agent news (bounded)
out1 = app.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Use one web search to find the 3 latest news items about AI agents. "
                    "For each item include: headline, source, date (if available), and one-line takeaway. "
                    "Then add a short overall findings summary (2-3 bullets) and include source URLs."
                )
            )
        ]
    },
    config=config,
)
print("TURN 1 FINAL:\n")
print(out1["messages"][-1].content)

# Turn 2 in the same thread (stateful memory)
out2 = app.invoke(
    {
        "messages": [
            HumanMessage(content="Now rewrite the findings summary for execs in 3 short bullets. No new search.")
        ]
    },
    config=config,
)
print("\nTURN 2 FINAL (same thread):\n")
print(out2["messages"][-1].content)

snapshot = app.get_state(config)
print("\nMessages currently in thread:", len(snapshot.values.get("messages", [])))

TURN 1 FINAL:

Here are the three latest news items about AI agents:

1. **Headline:** "AI agent arms race accelerates"  
   **Source:** Axios  
   **Date:** March 23, 2026  
   **Takeaway:** Companies are rapidly developing AI agent frameworks and safety measures in response to the rise of open-source models like OpenClaw.  
   **URL:** [Read more](https://www.axios.com/2026/03/23/openclaw-agents-nvidia-anthropic-perplexity)

2. **Headline:** "Gemini screen automation rolls out to Pixel 10"  
   **Source:** Android Central  
   **Date:** March 18, 2026  
   **Takeaway:** Google's Gemini now allows users to automate tasks on their Pixel 10 devices, enhancing user experience with AI-driven app interactions.  
   **URL:** [Read more](https://www.androidcentral.com/phones/google-pixel/gemini-screen-automation-expands-to-pixel-10-series)

3. **Headline:** "Nvidia launches Nemotron Coalition"  
   **Source:** Nvidia Newsroom  
   **Date:** March 16, 2026  
   **Takeaway:** Nvidia has formed

## TODO: tweak the same single tool

Inside `oai_web_search`, you can still keep one-tool architecture and tune behavior:

- Domain filtering:
  `tools=[{"type":"web_search","filters":{"allowed_domains":["openai.com","wikipedia.org"]}}]`
- Location-aware search:
  `user_location={"type":"approximate","country":"US","city":"New York","region":"New York"}`
- Cache-only mode:
  `external_web_access=False`